# Zero Forcing Sets and Network Controllability

**A-Level Mathematics Extension: Graph Theory and Network Control**

---

## Welcome!

In this notebook we explore a beautiful idea that connects graph theory to the engineering
problem of *controlling a network*. Imagine you have a collection of nodes connected by
edges — a social network, an electrical grid, a set of interacting robots. Which nodes
do you need to "control" (or "infect", or "activate") so that the whole network eventually
comes under your influence?

This is the **zero forcing** problem.

## Section 1: Introduction — What Is Zero Forcing?

### The Color-Change Rule

We start with a graph $G = (V, E)$ — a collection of *vertices* (nodes) and *edges*
(connections between nodes). We color each vertex either **blue** or **white**.

We choose some initial set $S \subseteq V$ to color blue. The remaining vertices start white.

We then repeatedly apply the **zero forcing color-change rule**:

> **If a blue vertex $v$ has exactly one white neighbor $u$, then $v$ *forces* $u$
> to become blue.**

We keep applying this rule until no more changes are possible.

- If every vertex is eventually colored blue, we call $S$ a **zero forcing set**.
- The **zero forcing number** $Z(G)$ is the size of the *smallest* zero forcing set.

### Why Does This Matter?

In control theory, the zero forcing number equals the **minimum number of external
inputs** needed to make a networked dynamical system *controllable* — meaning we can
steer it to any desired state. This was a surprising discovery connecting combinatorics
to engineering!

Let's build this up step by step.

## Section 2: Setup

We need three Python libraries:
- **NetworkX**: for creating and working with graphs
- **Matplotlib**: for drawing graphs and plots
- **NumPy**: for numerical computations

In [ ]:
# Install networkx quietly (it is pre-installed on Colab, but this ensures it)
import subprocess
subprocess.run(['pip', 'install', 'networkx', 'matplotlib', '--quiet'], capture_output=True)

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from itertools import combinations

print('Libraries loaded successfully!')
print(f'NetworkX version: {nx.__version__}')

## Section 3: The Zero Forcing Function

Let's implement the color-change rule in Python. The function below takes:
- `G`: a NetworkX graph
- `S`: the initial set of blue vertices

It returns:
- `blue`: the final set of blue vertices (after no more forcing is possible)
- `sequence`: the order in which vertices were forced blue

The key line is the check `if len(white_nbrs) == 1` — this is the forcing rule!

In [ ]:
def zero_force(G, S):
    """
    Apply zero forcing from initial set S on graph G.

    Parameters
    ----------
    G : networkx.Graph
        The graph on which to apply zero forcing.
    S : iterable
        The initial set of blue (forced) vertices.

    Returns
    -------
    blue : set
        The final set of blue vertices after all possible forcing steps.
    sequence : list
        The order in which vertices turned blue (initial set first,
        then each newly forced vertex in the order it was forced).
    """
    blue = set(S)          # Start with the initial blue set
    sequence = list(S)     # Record the order vertices turn blue
    changed = True

    while changed:
        changed = False
        new_blue = set()

        for v in blue:
            # Find all white (non-blue) neighbors of the blue vertex v
            white_nbrs = [u for u in G.neighbors(v) if u not in blue]

            # The forcing rule: if exactly ONE white neighbor, force it!
            if len(white_nbrs) == 1:
                new_blue.add(white_nbrs[0])

        if new_blue:
            blue |= new_blue                   # Add newly forced nodes to blue set
            sequence.extend(sorted(new_blue))  # Record the order
            changed = True                     # Keep looping

    return blue, sequence


# Quick test: path graph P5 with nodes {0, 1, 2, 3, 4}
G_test = nx.path_graph(5)
initial_blue = [0]  # Start with just the leftmost node blue

final_blue, order = zero_force(G_test, initial_blue)
print('Graph: P5 (path on 5 vertices)')
print(f'Initial blue set: {initial_blue}')
print(f'Final blue set:   {sorted(final_blue)}')
print(f'Order forced:     {order}')
print(f'Is zero forcing set? {len(final_blue) == len(G_test)}')

### Understanding the Output

On the path $P_5$ (vertices 0-1-2-3-4) starting with just vertex 0 blue:

- **Step 0**: Vertex 0 is blue. Its only white neighbor is 1 → **force vertex 1**.
- **Step 1**: Vertex 1 is now blue. Among its neighbors (0 and 2), only 2 is white → **force vertex 2**.
- **Step 2**: Vertex 2 forces vertex 3, and so on.

The single vertex $\{0\}$ is a zero forcing set for $P_5$! So $Z(P_5) = 1$.

This makes intuitive sense: a chain can be "taken over" starting from one end.

## Section 4: Visualising Zero Forcing on P₅ Step by Step

Let's make this visual! We'll draw the graph at each step of the forcing process,
coloring blue nodes in blue and white nodes in white.

In [ ]:
def plot_zero_forcing_steps(G, S, title='Zero Forcing on P₅', pos=None):
    """
    Visualize zero forcing step by step.
    Each subplot shows which nodes are blue after the next forcing step.
    """
    final_blue, sequence = zero_force(G, S)
    n_nodes = len(G)

    # Build the list of blue sets at each step
    # Step 0: only the initial set; Step k: first k+|S| nodes in sequence
    n_steps = len(sequence)  # Total steps = number of nodes eventually colored

    # Determine layout if not provided
    if pos is None:
        pos = nx.spring_layout(G, seed=42)

    # Create subplots: one per step
    fig, axes = plt.subplots(1, n_steps, figsize=(3 * n_steps, 3.5))
    if n_steps == 1:
        axes = [axes]  # Ensure axes is iterable for single subplot

    blue_so_far = set()
    for step_idx, node in enumerate(sequence):
        blue_so_far.add(node)
        ax = axes[step_idx]

        # Assign colors: blue for blue nodes, white for white nodes
        node_colors = [
            'royalblue' if v in blue_so_far else 'white'
            for v in G.nodes()
        ]

        nx.draw(
            G, pos=pos, ax=ax,
            node_color=node_colors,
            edgecolors='black',     # Border color of each node
            node_size=600,
            with_labels=True,
            font_color='white' if step_idx > 0 else 'white',
            font_weight='bold',
            width=2                 # Edge line width
        )

        # Title for each subplot
        if step_idx < len(S):
            ax.set_title(f'Initial\n(node {node} blue)', fontsize=10)
        else:
            ax.set_title(f'Step {step_idx - len(S) + 1}\nForce node {node}', fontsize=10)

    # Legend
    blue_patch = mpatches.Patch(color='royalblue', label='Blue (forced)')
    white_patch = mpatches.Patch(facecolor='white', edgecolor='black', label='White (unforced)')
    fig.legend(handles=[blue_patch, white_patch], loc='lower center', ncol=2, fontsize=10)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Create P5 and set a horizontal layout so it looks like a path
P5 = nx.path_graph(5)
pos_P5 = {i: (i, 0) for i in range(5)}  # Place nodes evenly on a horizontal line

plot_zero_forcing_steps(P5, [0], title='Zero Forcing on P₅ (path on 5 vertices)', pos=pos_P5)

The visualisation shows the "domino effect": once the leftmost node is blue, it
forces the next one, which forces the next one, and so on.

Notice that the font color of the labels changes — on a white node, we need dark text;
on a blue node, white text. This is a good example of how small details matter
in data visualisation!

## Section 5: Computing Z(G) for Several Graphs

Let's compute (or estimate) the zero forcing number for several classic graphs.
We'll try small initial sets and check whether they are zero forcing sets.

For small graphs we can be exhaustive: try all subsets of a given size and see
if any one of them forces all nodes blue.

In [ ]:
def zero_forcing_number(G):
    """
    Compute the zero forcing number Z(G) by trying all subsets.
    This is exhaustive and only practical for small graphs (|V| <= ~20).

    Returns
    -------
    z : int
        The zero forcing number Z(G).
    best_set : set
        One smallest zero forcing set achieving Z(G).
    """
    nodes = list(G.nodes())
    n = len(nodes)

    # Try subsets of increasing size until we find a zero forcing set
    for k in range(1, n + 1):
        for subset in combinations(nodes, k):
            forced, _ = zero_force(G, subset)
            if len(forced) == n:  # All nodes turned blue!
                return k, set(subset)

    return n, set(nodes)  # Fallback: all nodes (always a zero forcing set)


# ---- Define our test graphs ----
graphs = [
    ('Path P₅',       nx.path_graph(5)),
    ('Cycle C₆',      nx.cycle_graph(6)),
    ('Star S₄',       nx.star_graph(4)),
    ('Complete K₄',   nx.complete_graph(4)),
    ('Grid 3×3',      nx.grid_2d_graph(3, 3)),
]

print(f'{'Graph':<18} {'|V|':<6} {'|E|':<6} {'Z(G)':<8} {'A zero forcing set'}')
print('-' * 65)

results = []
for name, G in graphs:
    z, best_set = zero_forcing_number(G)
    nv = G.number_of_nodes()
    ne = G.number_of_edges()
    # Convert set to a sorted list for printing
    best_list = sorted(best_set, key=str)
    print(f'{name:<18} {nv:<6} {ne:<6} {z:<8} {best_list}')
    results.append((name, G, z, best_set))

### Interpreting the Results

Some patterns worth noting:

| Graph | Intuition for Z(G) |
|-------|--------------------|
| Path $P_n$ | $Z(P_n) = 1$: one endpoint forces the whole path |
| Cycle $C_n$ | $Z(C_n) = 2$: you need two adjacent blue nodes to break the symmetry |
| Star $K_{1,n}$ | $Z(K_{1,n}) = n-1$: you need all but one leaf blue (the centre forces the last) |
| Complete $K_n$ | $Z(K_n) = n-1$: you need all but one node (each node has many white neighbors, so forcing rarely applies) |
| Grid $3\times 3$ | $Z = 3$: an entire row forces the rest column by column |

**General rule for grids**: $Z(P_m \times P_n) = \min(m, n)$ — you need
an entire row or column as your initial set.

In [ ]:
# ---- Bar chart of results ----
fig, ax = plt.subplots(figsize=(8, 4))

names = [r[0] for r in results]
z_values = [r[2] for r in results]
n_values = [r[1].number_of_nodes() for r in results]

x = np.arange(len(names))
width = 0.35

bars1 = ax.bar(x - width/2, n_values, width, label='|V| (number of vertices)', color='steelblue', alpha=0.7)
bars2 = ax.bar(x + width/2, z_values, width, label='Z(G) (zero forcing number)', color='tomato', alpha=0.9)

ax.set_xlabel('Graph', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Zero Forcing Number Z(G) vs. Number of Vertices |V|', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Annotate bar heights
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9, color='darkred')

plt.tight_layout()
plt.show()

## Section 6: Visualising Zero Forcing on the 3×3 Grid

The 3×3 grid is more interesting than the path. Let's watch zero forcing
propagate when we start with the **entire top row** as our initial blue set.

On a grid, nodes are labeled as $(row, col)$ tuples. The top row is
$(0,0), (0,1), (0,2)$.

In [ ]:
# Create the 3x3 grid
G_grid = nx.grid_2d_graph(3, 3)

# Position: use the node labels directly as (x, y) coordinates
# Grid nodes are (row, col) tuples; we want (col, -row) so row 0 is at the top
pos_grid = {(r, c): (c, -r) for r in range(3) for c in range(3)}

# Initial blue set: entire top row
top_row = [(0, 0), (0, 1), (0, 2)]

# Run zero forcing and capture step-by-step state
final_blue, sequence = zero_force(G_grid, top_row)
print('Forcing sequence:', sequence)
print(f'Is zero forcing set? {len(final_blue) == len(G_grid)}')

# Build list of (node, blue_set_at_that_point) for unique stages
# Group the sequence into the initial set + individual forcing events
stages = []
current_blue = set()

# Stage 0: all top-row nodes at once
for n in top_row:
    current_blue.add(n)
stages.append(('Initial (top row blue)', set(current_blue)))

# Remaining stages: each newly forced node
for node in sequence[len(top_row):]:
    current_blue.add(node)
    stages.append((f'Force {node}', set(current_blue)))

# Plot
n_stages = len(stages)
cols = min(n_stages, 4)
rows = (n_stages + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = np.array(axes).flatten()  # Flatten to 1D for easy indexing

for i, (stage_title, blue_set) in enumerate(stages):
    ax = axes[i]
    node_colors = ['royalblue' if v in blue_set else 'white' for v in G_grid.nodes()]
    nx.draw(
        G_grid, pos=pos_grid, ax=ax,
        node_color=node_colors,
        edgecolors='black',
        node_size=700,
        with_labels=True,
        font_size=8,
        font_weight='bold',
        width=2
    )
    ax.set_title(stage_title, fontsize=10)

# Hide any unused subplots
for i in range(n_stages, len(axes)):
    axes[i].set_visible(False)

# Legend
blue_patch = mpatches.Patch(color='royalblue', label='Blue (forced)')
white_patch = mpatches.Patch(facecolor='white', edgecolor='black', label='White (unforced)')
fig.legend(handles=[blue_patch, white_patch], loc='lower center', ncol=2, fontsize=10)

fig.suptitle('Zero Forcing on 3×3 Grid — Starting from Top Row', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### What's Happening on the Grid?

Watch the forcing propagate **row by row**:

1. Top row $(0,0), (0,1), (0,2)$ is blue initially.
2. Each top-row node has exactly one white neighbor below it, so it forces
   the middle row: $(1,0), (1,1), (1,2)$ turn blue.
3. Now the middle row forces the bottom row in the same way.

This is why $Z(3 \times 3 \text{ grid}) = 3$ — we need at least a full row
(or column) to start the cascade.

## Section 7: Challenge Exercise — Implementing `zero_forcing_number`

We've already used `zero_forcing_number` above, but let's look at it more carefully
and think about the algorithm.

**Your task**: The skeleton below has some gaps marked `# YOUR CODE HERE`.
Try to fill them in before looking at the solution!

In [ ]:
# ---- Challenge: implement zero_forcing_number from scratch ----
#
# Hint: use the combinations() function from itertools to generate
# all subsets of size k.
#
# combinations([0,1,2,3], 2) gives: (0,1), (0,2), (0,3), (1,2), (1,3), (2,3)

def zero_forcing_number_challenge(G):
    """
    Find the zero forcing number of G.

    Strategy:
      For k = 1, 2, 3, ..., |V|:
        For each subset S of vertices with |S| = k:
          Run zero_force(G, S)
          If all vertices are blue, return k
    """
    nodes = list(G.nodes())
    n = len(nodes)

    for k in range(1, n + 1):
        # Iterate over all subsets of 'nodes' of size k
        for subset in combinations(nodes, k):       # <-- key step
            forced, _ = zero_force(G, subset)       # Run the forcing rule
            if len(forced) == n:                    # Did we color everything?
                return k, set(subset)               # Yes! Return the answer

    # This line should never be reached for a connected graph
    return n, set(nodes)


# ---- Test it ----
test_graphs = [
    ('P₄', nx.path_graph(4)),
    ('C₅', nx.cycle_graph(5)),
    ('K₃', nx.complete_graph(3)),
]

print('Testing zero_forcing_number_challenge:')
print(f'{'Graph':<10} {'Z(G)':<8} {'A smallest ZF set'}')
print('-' * 40)
for name, G in test_graphs:
    z, s = zero_forcing_number_challenge(G)
    print(f'{name:<10} {z:<8} {sorted(s, key=str)}')

print()
print('Complexity note:')
print('For a graph with n vertices, we check up to C(n,1)+C(n,2)+... subsets.')
print('This is 2^n in the worst case — exponential! For large graphs we need')
print('cleverer algorithms (this is actually an NP-hard problem in general).')

### Complexity Note

Computing $Z(G)$ exactly requires trying all subsets, and there are $2^n$ subsets
of an $n$-vertex graph. For $n = 20$, that's over one million subsets!

In fact, computing $Z(G)$ is **NP-hard** in general — meaning there's no known
efficient algorithm that works for all graphs. For special families (paths, trees,
cycles, grids), we have exact formulas.

This is a common theme in combinatorics and computer science: easy to define,
but hard to compute!

## Section 8: Connection to Controllability

### Why Does Zero Forcing Appear in Control Theory?

Consider a network of $n$ agents (robots, neurons, power grid nodes) whose
state evolves according to the **linear dynamical system**:

$$\dot{x}(t) = A x(t) + B u(t)$$

where:
- $x(t) \in \mathbb{R}^n$ is the state of all agents at time $t$
- $A \in \mathbb{R}^{n \times n}$ encodes how agents influence each other
  (the **adjacency structure** of the graph!)
- $u(t) \in \mathbb{R}^m$ is our external control input
- $B$ selects which agents we directly control

The system is **controllable** if we can steer it from any state to any other
state using the inputs $u(t)$. The question is: **what is the minimum number
of directly controlled agents $m$ needed?**

### The Surprising Connection

For a generic choice of edge weights on graph $G$, the **minimum number of
inputs for controllability equals $Z(G)$**, the zero forcing number!

This was proved in 2007 and was quite surprising: a combinatorial graph property
exactly characterizes a linear-algebraic control property.

The zero forcing rule captures an "information flow" argument:
- A blue node = a node whose state is "known" or "determined"
- Forcing = if a node's state is determined and has only one unknown neighbor,
  the neighbor's state can be deduced

Let's visualize this connection with a small example.

In [ ]:
# Visualize the connection between zero forcing and controllability
# We'll show a graph and its adjacency matrix side by side

G = nx.path_graph(4)  # P4: 0 - 1 - 2 - 3
pos = {i: (i, 0) for i in range(4)}

# Zero forcing set for P4 is {0} (one endpoint)
zf_set = {0}
final, seq = zero_force(G, zf_set)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: draw the graph, highlighting the ZF set
ax = axes[0]
node_colors = ['royalblue' if v in zf_set else 'white' for v in G.nodes()]
nx.draw(
    G, pos=pos, ax=ax,
    node_color=node_colors,
    edgecolors='black',
    node_size=700,
    with_labels=True,
    font_weight='bold',
    width=3
)
ax.set_title('Graph P₄\nBlue node = directly controlled\n(Z(P₄) = 1 input needed)', fontsize=11)

# Right: show the adjacency matrix
ax2 = axes[1]
A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
im = ax2.imshow(A, cmap='Blues', vmin=0, vmax=1)
ax2.set_xticks(range(4))
ax2.set_yticks(range(4))
ax2.set_xticklabels([f'v{i}' for i in range(4)])
ax2.set_yticklabels([f'v{i}' for i in range(4)])
ax2.set_title('Adjacency Matrix of P₄\n(Structure matrix A in ẋ = Ax + Bu)', fontsize=11)

# Annotate matrix entries
for i in range(4):
    for j in range(4):
        ax2.text(j, i, int(A[i, j]), ha='center', va='center',
                 fontsize=14, color='white' if A[i,j] > 0.5 else 'black')

plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
fig.suptitle('Zero Forcing Number = Minimum Inputs for Controllability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nSummary:')
print(f'  Graph: P₄  |  Z(P₄) = 1')
print(f'  Meaning: controlling just ONE node (e.g. v0) is enough')
print(f'  to make the entire path-network controllable!')
print(f'  Forcing sequence: {seq}')

## Summary and Key Takeaways

In this notebook we explored **zero forcing** — a graph-theoretic concept with
beautiful mathematical properties and real engineering applications.

### What We Learned

1. **The color-change rule**: A blue vertex with exactly one white neighbor forces
   that neighbor to turn blue.

2. **Zero forcing sets**: An initial set $S$ from which the entire graph can be forced blue.

3. **The zero forcing number** $Z(G)$: the minimum size of a zero forcing set.

4. **Examples**:
   - $Z(P_n) = 1$ (paths need one endpoint)
   - $Z(C_n) = 2$ (cycles need two adjacent nodes)
   - $Z(K_n) = n-1$ (complete graphs need all but one node)
   - $Z(P_m \times P_n) = \min(m,n)$ (grids need a full row or column)

5. **Connection to control theory**: $Z(G)$ = minimum number of external inputs
   needed to make a linear network dynamical system controllable.

### Further Exploration

- Can you find the zero forcing number of the Petersen graph?
- What happens to $Z(G)$ when you add edges to $G$? Does it always decrease?
- Research **propagation time** — the number of steps until all nodes are forced.